In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [4]:
import pandas as pd
import datetime
from datetime import timedelta

def preprocess_hourly_data(): # 시간별 데이터 전처리
    df = pd.read_csv("data/asos_seoul_hourly.csv") #시간별 데이터 읽기
    df['tm'] = pd.to_datetime(df['tm'])

    flood_periods = [ #실제 침수 사건
        ("2000-08-23", "2000-09-01"),
        ("2002-08-30", "2002-09-01"),
        ("2005-08-02", "2005-08-11"),
        ("2006-07-09", "2006-07-29"),
        ("2007-09-13", "2007-09-13"),
        ("2011-07-26", "2011-07-29"),
        ("2013-07-11", "2013-07-15"),
        ("2013-07-18", "2013-07-18"),
        ("2018-08-23", "2018-08-24"),
        ("2018-08-26", "2018-09-01"),
        ("2019-09-28", "2019-10-03"),
        ("2020-07-28", "2020-08-11"),
        ("2020-08-28", "2020-09-03"),
        ("2020-09-01", "2020-09-07"),
        ("2022-08-08", "2022-08-17"),
        ("2022-08-28", "2022-09-06")
    ]

    flood_dates = set() #홍수 발생 시간들을 저장할 셋
    for start, end in flood_periods:
        date_range = pd.date_range(start=start, end=end, freq='H') #각 홍수 기간에 대해 시작일부터 종료일까지 시간별(freq='H') 범위를 생성(시간별 처리)
        flood_dates.update(date_range) #모든 홍수 발생 시간들을 set에 추가

    df['flood_risk'] = df['tm'].isin(flood_dates).astype(int) #각 시간이 홍수 발생 시간에 해당하는지 확인
    df.to_csv("data/asos_seoul_hourly_with_flood_risk.csv", index=False) #csv로 저장



In [5]:
def preprocess_daily_data(): #일별 데이터 전처리
    df = pd.read_csv("data/asos_seoul_daily.csv")
    df['tm'] = pd.to_datetime(df['tm'])
    #날짜에서 월, 요일, 연도 정보를 추출하여 새로운 컬럼으로 생성
    df['month'] = df['tm'].dt.month #월
    df['dayofweek'] = df['tm'].dt.dayofweek #요일
    df['year'] = df['tm'].dt.year #연도
    df['sumRn'] = df['sumRn'].fillna(0) #강수량: 결측값을 0으로 대체

    median_cols = ['minTa', 'maxTa', 'avgWs', 'avgTs', 'sumGsr', 'maxInsWs', 'sumSmlEv', 'avgPs'] #온도, 풍속, 일사량 등 8개 컬럼: 결측값을 각 컬럼의 중간값으로 대체
    for col in median_cols:
        df[col] = df[col].fillna(df[col].median())

    df['ddMefs'] = df['ddMefs'].fillna(0) #ddMefs: 결측값을 0으로 대체

    flood_periods = [ #실제 침수 사건
        ("2000-08-23", "2000-09-01"),
        ("2002-08-30", "2002-09-01"),
        ("2005-08-02", "2005-08-11"),
        ("2006-07-09", "2006-07-29"),
        ("2007-09-13", "2007-09-13"),
        ("2011-07-26", "2011-07-29"),
        ("2013-07-11", "2013-07-15"),
        ("2013-07-18", "2013-07-18"),
        ("2018-08-23", "2018-08-24"),
        ("2018-08-26", "2018-09-01"),
        ("2019-09-28", "2019-10-03"),
        ("2020-07-28", "2020-08-11"),
        ("2020-08-28", "2020-09-03"),
        ("2020-09-01", "2020-09-07"),
        ("2022-08-08", "2022-08-17"),
        ("2022-08-28", "2022-09-06")
    ]

    flood_dates = set() #홍수 발생 일들을 저장할 셋
    for start, end in flood_periods: #시작일부터 종료일까지 하루씩 증가시키며 날짜들을 set에 추가(일별 처리)
        start = datetime.datetime.strptime(start, "%Y-%m-%d")
        end = datetime.datetime.strptime(end, "%Y-%m-%d")
        while start <= end:
            flood_dates.add(start.date())
            start += timedelta(days=1) #하루씩 증가

    df['flood_risk'] = df['tm'].dt.date.isin(flood_dates).astype(int) #flood_risk : 1이면 침수 발생, 0이면 침수 미발생
    df.loc[df['sumRn'] >= 30, 'flood_risk'] = 1 #강수량이 30mm 이상이면 침수 발생(1)

    df.to_csv("data/asos_seoul_daily_enriched.csv", index=False) #csv로 저장

In [6]:
def preprocess_xgboost_features(): #모델 학습을 위해 기존 일별 기상 데이터에 월/일/요일/주말여부/강우관련 파생변수들을 추가로 생성하는 함수
    import pandas as pd

    df = pd.read_csv("data/asos_seoul_daily_enriched.csv") #일별 전처리 데이터 불러오기
    df['tm'] = pd.to_datetime(df['tm'])

    df['month'] = df['tm'].dt.month #월
    df['day'] = df['tm'].dt.day #일
    df['weekday'] = df['tm'].dt.weekday #요일
    df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0) #주말인지 여부(5, 6). 주말이면1, 아니면0

    df['is_rainy'] = df['sumRn'].apply(lambda x: 1 if x >= 30 else 0) #강수량이 30mm 이상이면 침수 발생(1)
    df['rain_hours'] = df['sumRn'].apply(lambda x: round(x / 3)) #강수량을 3으로 나눈 값을 반올림 (대략적인 강우 시간 추정)
    df['max_hourly_rn'] = df['sumRn'].apply(lambda x: x if x <= 50 else 50) #강수량이 50을 초과하면 50으로 제한

    df.to_csv("data/asos_seoul_daily_enriched.csv", index=False)
    print("XGBoost용 파생 변수 추가 완료 및 저장됨.")